# FID Evaluation with pytorch-fid

Quick and simple FID evaluation using the pytorch-fid library.

In [5]:
import torch
import torch.nn as nn
from fid_simple import compute_fid_from_checkpoint, evaluate_all_checkpoints_simple

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Install pytorch-fid if not already installed
import subprocess
import sys
try:
    import pytorch_fid
except ImportError:
    print("Installing pytorch-fid...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytorch-fid"])

Device: cuda


In [6]:
# Generator classes
class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 64 * 8 * 8),
            nn.ReLU(),

            nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(True),

            nn.Conv2d(16, 3, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        output = self.model(x)
        return output
    
class WGANGPGenerator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 512 * 4 * 4),
            nn.ReLU(True),

            nn.Unflatten(1, (512, 4, 4)),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1,bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1,bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            nn.Conv2d(32, 3, 3, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        output = self.model(x)
        return output

In [7]:
# Evaluate all checkpoints
# Both DCGAN and WGAN-GP were trained with latent_dim=100
results = evaluate_all_checkpoints_simple(
    checkpoint_dirs={
        'DCGAN': 'checkpoints/dcgan',
        'WGAN-GP': 'checkpoints/wgan'
    },
    generator_classes={
        'DCGAN': DCGANGenerator,
        'WGAN-GP': WGANGPGenerator
    },
    real_images_dir='data/villagers',
    latent_dim=100,
    num_samples=500,
    device=device
)


Evaluating DCGAN

epoch_epoch_1000_DCGAN_discriminator_leaky_relu_0_2_sigmoidEnd_Adam_0_5-0_999_generator_relu_TanhEnd_Adam_0_5-0_999_2026-03-05_18-14-10_checkpoint.pth
Loading checkpoint: checkpoints/dcgan\epoch_epoch_1000_DCGAN_discriminator_leaky_relu_0_2_sigmoidEnd_Adam_0_5-0_999_generator_relu_TanhEnd_Adam_0_5-0_999_2026-03-05_18-14-10_checkpoint.pth
Generating 500 images...


Computing FID score...


  0%|          | 0/13 [00:03<?, ?it/s]


✗ Error: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmer

Computing FID score...


  0%|          | 0/13 [00:03<?, ?it/s]

✗ Error: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmer

In [8]:
# Or evaluate a single checkpoint
fid_score = compute_fid_from_checkpoint(
    checkpoint_path='checkpoints/dcgan/epoch_epoch_1000_DCGAN_discriminator_leaky_relu_0_2_sigmoidEnd_Adam_0_5-0_999_generator_relu_TanhEnd_Adam_0_5-0_999_2026-03-05_18-14-10_checkpoint.pth',
    generator_class=DCGANGenerator,
    real_images_dir='data/villagers',
    num_samples=500,
    latent_dim=100,
    device=device
)

print(f"FID Score: {fid_score:.4f}")

Loading checkpoint: checkpoints/dcgan/epoch_epoch_1000_DCGAN_discriminator_leaky_relu_0_2_sigmoidEnd_Adam_0_5-0_999_generator_relu_TanhEnd_Adam_0_5-0_999_2026-03-05_18-14-10_checkpoint.pth
Generating 500 images...


Computing FID score...


  0%|          | 0/13 [00:03<?, ?it/s]


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Programmering\Computer Science\AdvancedMachineLearning\AML-2026-Miniproject\.venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\DocumentsGlobal\Progr